In [3]:
import pandas as pd
import glob

csv_files = glob.glob(r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\*.csv")
datasets = {}

for file in csv_files:
    data = pd.read_csv(file)
    datasets[file] = data


In [5]:
customers = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_customers_dataset.csv"]
geolocation = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_geolocation_dataset.csv"]
order_items = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_order_items_dataset.csv"]
order_payments = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_order_payments_dataset.csv"]
order_reviews = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_order_reviews_dataset.csv"]
orders = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_orders_dataset.csv"]
products = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_products_dataset.csv"]
sellers = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\olist_selllers_dataset.csv"]
category_translation = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\cleaned\product_category_name_translation.csv"]

In [6]:
len(datasets)

9

In [7]:
geolocation.duplicated().sum()

np.int64(0)

In [9]:
geolocation.shape

(738332, 5)

In [12]:
products.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [15]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

In [16]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [17]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [21]:
review_date_columns = ["review_creation_date","review_answer_timestamp"]

order_reviews[review_date_columns] = order_reviews[review_date_columns].apply(pd.to_datetime)

In [22]:
order_reviews[review_date_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [23]:
order_reviews[review_date_columns].isna().sum()

review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [24]:
delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

len(delivered_missing_date)

8

In [25]:
delivered_missing_date[
    [
        "order_id",
        "order_status",
        "order_delivered_customer_date"
    ]
]

,order_id,order_status,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,NaT
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,NaT
98038,20edc82cf5400ce95e1afacc25798b31,delivered,NaT


In [26]:
duplicate_reviews = order_reviews[order_reviews["review_id"].duplicated(keep=False)]

duplicate_reviews["review_id"].nunique()

789

In [27]:
len(duplicate_reviews)

1603

In [28]:
zero_installment = order_payments[(order_payments["payment_type"] == "credit_card") &order_payments["payment_installments"] == 0)]

len(zero_installment)

2

In [29]:
zero_installment[["order_id","payment_type","payment_value","payment_installments"]]

,order_id,payment_type,payment_value,payment_installments
46982,744bade1fcf9ff3f31d860ace076d422,credit_card,58.69,0
79014,1a57108394169c0b47d8f876acc9ba2d,credit_card,129.94,0


In [31]:
missing_category = products[products["product_category_name"].isna()]

len(missing_category)

610

In [32]:
missing_dimensions = products[products[["product_weight_g","product_length_cm","product_height_cm","product_width_cm"]].isna().all(axis=1)]

len(missing_dimensions)

2

In [33]:
orders["order_id"].is_unique

True

In [34]:
products["product_id"].is_unique

True

In [35]:
sellers["seller_id"].is_unique

True

In [36]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

## Validation Conclusion

I validated the cleaned datasets against the decisions made during the quality assessment.

I confirmed that:

- Exact geolocation duplicates were removed.
- The expected product column names are present.
- Order and review date columns can be converted to datetime.
- The 8 delivered orders with missing delivery dates were preserved.
- Repeated review IDs were preserved.
- The 2 zero-installment credit-card records were preserved.
- Missing product categories and dimensions were preserved.
- Key identifiers and the `order_id` + `order_item_id` relationship remain valid.

The cleaned data passed the validation checks and is ready for the next stage: SQLite database creation.